# Software profesional en Acústica 2025-26 (M2i)

*This notebook contains a modification of the notebook [FEM_example_meshes_2D](https://github.com/spatialaudio/computational_acoustics/blob/master/FEM_example_meshes_2D.ipynb), created by Sascha Spors, Frank Schultz, Computational Acoustics Examples, 2018. The text/images are licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/). The code is released under the [MIT license](https://opensource.org/licenses/MIT).*

First, we need to install on the fly [NGSolve](https://ngsolve.org/) using the [FEM on Colab](https://fem-on-colab.github.io/packages.html) install script:

In [1]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# Two-Dimensional Example Meshes

This notebook shows some examples for two-dimensional meshes as used in the [Finite Element Method](https://en.wikipedia.org/wiki/Finite_element_method) (FEM) for spatial discretization. The meshes are generated by [NGSolve](https://ngsolve.org/) and [Netgen](https://ngsolve.org/), an open-source framework for numerical solution of PDEs.

In [2]:
import ngsolve
from netgen.geom2d import SplineGeometry
from ngsolve.meshes import MakeStructured2DMesh
from ngsolve.webgui import Draw

## Rectangular Domains

In [3]:
# Rectangle from (-2,-2) to (2,2) with ~20 elements per side
mesh = MakeStructured2DMesh(quads=False, nx=20, ny=20, mapping = lambda x,y : (4*x-2, 4*y-2))
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

In [4]:
# Rectangle from (-2,-2) to (2,2) with mesh density ~20 => maxh ~ 4/20 = 0.2
geo = SplineGeometry()
geo.AddRectangle((-2, -2), (2, 2), bc="boundary")
ngmesh = geo.GenerateMesh(maxh=0.2)
mesh = ngsolve.Mesh(ngmesh)
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

## Coupled Rectangular Domains

In [5]:
# Three connected rectangles: left (0,0)-(3,4), connector (3,1.5)-(3.5,2.5), right (3.5,0)-(6,4)
# We build the union geometry manually as a single connected polygon
geo = SplineGeometry()

# Add all points of the outer boundary of the union
# Left rectangle outer points (going CCW), then connector, then right rectangle
pts = [
    geo.AppendPoint(0.0, 0.0),   # 0
    geo.AppendPoint(3.0, 0.0),   # 1  (left rect bottom-right = connector bottom-left)
    geo.AppendPoint(3.0, 1.5),   # 2  (connector bottom-left of narrow part)
    geo.AppendPoint(3.5, 1.5),   # 3  (connector bottom-right -> right rect)
    geo.AppendPoint(3.5, 0.0),   # 4  (right rect bottom-left)
    geo.AppendPoint(6.0, 0.0),   # 5  (right rect bottom-left after connector)
    geo.AppendPoint(6.0, 4.0),   # 6  (right rect top-right)
    geo.AppendPoint(3.5, 4.0),   # 7
    geo.AppendPoint(3.5, 2.5),   # 8  (connector top-right)
    geo.AppendPoint(3.0, 2.5),   # 9  (connector top-left)
    geo.AppendPoint(3.0, 4.0),   # 10
    geo.AppendPoint(0.0, 4.0),   # 11
]

# Connect points to form the boundary
segments = [
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 5),
    (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 0)
]
for a, b in segments:
    geo.Append(["line", pts[a], pts[b]], leftdomain=1, rightdomain=0, bc="boundary")

ngmesh = geo.GenerateMesh(maxh=0.3)
mesh = ngsolve.Mesh(ngmesh)
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

## Domains with Holes

In [6]:
# Rectangle (-4,-4)-(4,4) with a rectangular hole (-.75,-.75)-(.75,.75)
geo = SplineGeometry()

# Outer rectangle: leftdomain=1, rightdomain=0
geo.AddRectangle((-4, -4), (4, 4), leftdomain=1, rightdomain=0, bc="outer")

# Inner rectangle (hole): leftdomain=0, rightdomain=1
geo.AddRectangle((-0.75, -0.75), (0.75, 0.75), leftdomain=0, rightdomain=1, bc="inner")

ngmesh = geo.GenerateMesh(maxh=0.4)
mesh = ngsolve.Mesh(ngmesh)
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

In [7]:
# Rectangle (-4,-4)-(4,4) with a circular hole of radius 1 at origin
geo = SplineGeometry()

# Outer rectangle: leftdomain=1, rightdomain=0
geo.AddRectangle((-4, -4), (4, 4), leftdomain=1, rightdomain=0, bc="outer")

# Inner circle (hole): leftdomain=0, rightdomain=1
geo.AddCircle((0, 0), r=1, leftdomain=0, rightdomain=1, bc="inner")

ngmesh = geo.GenerateMesh(maxh=0.4)
mesh = ngsolve.Mesh(ngmesh)
mesh.Curve(2)  # Curve order 2 for better approximation of the circle
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

## Polygonial Domains

In [8]:
# Polygonal domain defined by a list of vertices
domain_vertices = [
    (0.0, 0.0),
    (5.0, 0.0),
    (5.0, 0.75),
    (4.5, 0.75),
    (4.0, 1.50),
    (2.0, 1.50),
    (1.0, 0.75),
    (0.0, 0.75),
]

geo = SplineGeometry()
pts = [geo.AppendPoint(x, y) for x, y in domain_vertices]

# Close the polygon: connect consecutive points, and close last->first
n = len(pts)
for i in range(n):
    geo.Append(["line", pts[i], pts[(i + 1) % n]],
               leftdomain=1, rightdomain=0, bc="boundary")

# maxh ~ 5/20 = 0.25 to approximate 20-element density
ngmesh = geo.GenerateMesh(maxh=0.25)
mesh = ngsolve.Mesh(ngmesh)
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).